<a href="https://colab.research.google.com/github/eeeewyz/LLM/blob/main/4_prompt_engineering_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ungraded Lab: Prompt Engineering


## 1. Introduction

Welcome to the ungraded lab on Prompt Engineering! In this lab you will explore some prompt techniques to help adjust the LLM to your needs. Mainly in this lab you will:


1. Learn how to make an LLM generate specific outputs, such as labeling a sentence
2. Make an LLM call with different parameters depending on the task nature of the prompt
3. Make an LLM return a specific object type in its response, like a JSON


### 1.1 Importing the libraries


# Table of Contents
- [ 1 - Text Classification with LLMs](#1)
- [ 2 - Parameter Setting Based on Tasks](#2)
- [ 3 - Guiding the LLM to Output Specific Objects](#3)


In [ ]:
from utils import (
    generate_with_single_input,
    generate_with_multiple_input,
    generate_params_dict
)

<a id='1'></a>
## 1 - Text Classification with LLMs

An interesting and practical application of language models (LLMs) is transforming them into text classifiers. With good instructions, you can guide an LLM to categorize text based on sentiment, task type, and more. Since classifiers typically output fixed labels (such as 1 for positive and 0 for negative), you need to construct a prompt that minimizes the likelihood of the LLM producing unexpected outputs. Besides designing a robust prompt, implementing checks in your code is something to consider to avoid potential issues, such as a function later in the process expecting values of 1 or 0 but receiving an unexpected value like 2 or a phrase such as "positive sentence." This combination of strategies ensures reliable classification results while maintaining flexibility in handling unforeseen outputs.

To illustrate, let's suppose you are developing a chatbot for a company that sells sport outfits and nutritional supplements. The idea is to make an LLM decide if the query is related to outfits or nutrition. This might be useful to redirect your LLM to the correct database to query from.

1. Be precise. You need to explain exactly what you want it to do and output.
2. Add examples. Create examples with their expected result.
3. You might add also edgy examples, i.e., examples that you know that might be hard for the LLM to properly decide.

In [ ]:
def check_if_outfit_or_supplement(query):
    """
    根据用户输入的 query，构造一个用于分类的 Prompt。

    分类结果只有两种：
    1. Nutritional：营养品、补剂、健康食品等
    2. Outfit：服装、鞋子、饰品、时尚产品等

    Args:
        query (str): 用户输入的问题或搜索内容。

    Returns:
        str: 构造好的完整 Prompt。
             注意：这个函数只负责生成 Prompt，
             并不会直接调用 LLM，也不会直接返回分类结果。
    """

    # 使用 f-string 构造 Prompt，
    # 这样可以把函数参数 query 插入到 Prompt 中
    prompt = f"""
Determine the category of the following query as either "nutritional" or "outfit" related.

# 定义 Nutritional 类别：
# 与营养产品、蛋白粉、维生素、补剂、健康食品和饮料有关
- Nutritional queries: These are related to nutrition products, such as whey protein, vitamins, supplements, dietary products, and health-related food and beverages.

# 定义 Outfit 类别：
# 与服装、鞋子、配饰、珠宝和时尚有关
- Outfit queries: These pertain to clothing and fashion, including items like shirts, dresses, shoes, accessories, and jewelry.

# 提供分类示例，帮助模型理解分类标准
Examples:

1. Query: “Where can I buy high-protein snacks?” Expected answer: Nutritional
2. Query: “Best shirt styles for summer 2023” Expected answer: Outfit
3. Query: “Are there any shoes designed for running?” Expected answer: Outfit
4. Query: “What multivitamins should I take daily?” Expected answer: Nutritional
5. Query: “Best weight loss products that are stylish” Expected answer: Nutritional
6. Query: “Athletic wear that boosts performance” Expected answer: Outfit

# 将用户实际输入的内容插入这里
Query: {query}

# 限制模型的输出格式：
# 只能回答 Nutritional 或 Outfit
Instructions: Respond with “Nutritional” if the query pertains to nutritional products or “Outfit” if it pertains to clothing or fashion products.

# 要求只输出一个单词，避免模型给出解释
Answer only one single word.
"""

    # 返回构造好的 Prompt 字符串
    return prompt

In [ ]:
# Testing a simple query
query = "Give me the available vitamins supplement you have in your catalogue."
generate_with_single_input(check_if_outfit_or_supplement(query), max_tokens = 2)

{'role': 'assistant', 'content': 'Nutritional'}

准备多条测试问题
        ↓
逐条生成分类 Prompt
        ↓
调用 LLM 进行分类
        ↓
获得 Nutritional 或 Outfit
        ↓
与人工正确答案比较
        ↓
正确显示绿色，错误显示红色

Now let's test in a bigger set.

In [ ]:
# 定义终端输出颜色的 ANSI 转义码
# GREEN：绿色，用于表示预测正确
GREEN = '\033[92m'

# RED：红色，用于表示预测错误
RED = '\033[91m'

# RESET：恢复终端默认颜色，避免后面的文字继续带颜色
RESET = '\033[0m'


# 构造测试数据集
# 每个字典包含：
# query：需要分类的用户问题
# label：人工标注的正确分类结果
queries = [
    {"query": "Where can I buy whey protein?", "label": "Nutritional"},
    {"query": "Recommended vitamins for winter", "label": "Nutritional"},
    {"query": "Latest fashion for women's dresses", "label": "Outfit"},
    {"query": "Comfortable sneakers for daily use", "label": "Outfit"},
    {"query": "Best energy bars for athletes", "label": "Nutritional"},
    {"query": "Trendy accessories for men", "label": "Outfit"},
    {"query": "Low-carb diet food options", "label": "Nutritional"},
    {"query": "What supplements help with muscle recovery?", "label": "Nutritional"},
    {"query": "Casual wear that supports healthy living", "label": "Outfit"}
]


# 逐个处理测试数据中的每一条内容
for item in queries:

    # 取出当前测试问题
    query = item["query"]

    # 把当前问题放入分类 Prompt 模板中
    # 这里只是生成 Prompt，还没有调用模型
    prompt = check_if_outfit_or_supplement(query)

    # 取出人工标注的正确答案
    expected_label = item["label"]

    # 将完整 Prompt 发送给 LLM
    # max_tokens=2 表示最多生成两个 token
    response = generate_with_single_input(
        prompt,
        max_tokens=2
    )

    # 从模型返回的字典中提取真正的回答
    # 例如：
    # {
    #     "role": "assistant",
    #     "content": "Nutritional"
    # }
    result = response["content"]

    # 比较模型预测结果与正确标签
    if result == expected_label:
        # 如果完全相同，使用绿色
        color = GREEN
    else:
        # 如果不相同，使用红色
        color = RED

    # 输出当前问题、模型预测结果和正确答案
    print(
        f"Query: {query}\n"
        f"Result: {result}\n"
        f"Expected: {color}{expected_label}{RESET}\n"
    )

Query: Where can I buy whey protein?
Result: Nutritional
Expected: Nutritional

Query: Recommended vitamins for winter
Result: Nutritional
Expected: Nutritional

Query: Latest fashion for women's dresses
Result: Outfit
Expected: Outfit

Query: Comfortable sneakers for daily use
Result: Outfit
Expected: Outfit

Query: Best energy bars for athletes
Result: Nutritional
Expected: Nutritional

Query: Trendy accessories for men
Result: Outfit
Expected: Outfit

Query: Low-carb diet food options
Result: Nutritional
Expected: Nutritional

Query: What supplements help with muscle recovery?
Result: Nutritional
Expected: Nutritional

Query: Casual wear that supports healthy living
Result: Nutritional
Expected: Outfit



<a id='2'></a>
## 2 - Parameter Setting Based on Tasks

In this section, you will learn how to adjust your LLM interactions to be flexible, allowing you to control its behavior based on the nature of the task. This involves determining the nature of the query before requesting the LLM to respond.

In this exercise, let's develop a function to categorize a query as either technical or creative. Once categorized, you can apply different parameters suited for each task type. Technical queries generally benefit from lower randomness, whereas creative tasks may benefit from allowing higher randomness.

In [ ]:
def decide_if_technical_or_creative(query):
    """
    Determines whether a given query is creative or technical in nature.

    Args:
        query (str): The query string to be evaluated.

    Returns:
        str: A label indicating the query type, either 'creative' or 'technical'.

    This function constructs a prompt to classify a query based on its content.
    Creative queries typically involve requests to generate original content, whereas
    technical queries relate to documentation or technical information, such as procedures.
    By leveraging an LLM, it identifies the query type and returns an appropriate label.
    """

    PROMPT = f"""Decide if the following query is a creative query or a technical query.
    Creative queries ask you to create content, while technical queries are related to documentation or technical requests, like information about procedures.
    Answer only 'creative' or 'technical'.
    Query: {query}
    """
    result = generate_with_single_input(PROMPT)
    label = result['content']
    return label

In [ ]:
queries = ["What is Pi-hole?",
           "Suggest to me three places to visit in South America"]
for query in queries:
    label =decide_if_technical_or_creative(query)
    print(f"Query: {query}, label: {label}")

Query: What is Pi-hole?, label: technical
Query: Suggest to me three places to visit in South America, label: creative


接收用户问题
   ↓
判断问题类型
   ↓
根据类型设置 temperature 和 top_p
   ↓
调用大语言模型
   ↓
提取回答内容并返回

In [ ]:
def answer_query(query):
    """
    Processes a query and generates an appropriate response by categorizing the query
    as either 'technical' or 'creative', and modifies behavior based on this categorization.

    Args:
        query (str): The query string to be answered.

    Returns:
        str: A generated response from the LLM tailored to the nature of the query.

    This function first determines the nature of the query using the `decide_if_technical_or_creative` function.
    If the query is classified as 'technical', it sets parameters suitable for precise and low-variability responses.
    If the query is 'creative', it applies parameters allowing for more variability and creativity.
    If the classification is inconclusive, it uses neutral parameters.
    It then generates a response using these parameters and returns the content.
    """

    # Determine whether the query is 'technical' or 'creative'
    label = decide_if_technical_or_creative(query).lower()

    # Set parameters for technical queries (precise, low randomness)
    if label == 'technical':
        kwargs = generate_params_dict(query, temperature=0, top_p=0.1)

    # Set parameters for creative queries (variable, high randomness)
    elif label == 'creative':
        kwargs = generate_params_dict(query, temperature=1.1, top_p=0.4)

    # Use default parameters if the query type is inconclusive
    else:
        kwargs = generate_params_dict(query, temperature=0.5, top_p=0.5)

    # Generate a response based on the query type and parameters
    response = generate_with_single_input(**kwargs)

    # Extract and return the content from the response
    result = response['content']
    return result

In [ ]:
queries = ["What is Pi-hole?",
           "Suggest to me three places to visit in South America"]
for query in queries:
    result = answer_query(query)
    print(f"Query: {query}\nAnswer: {result}\n\n#######\n")

Query: What is Pi-hole?
Answer: **Pi-hole** is a free, open-source, **network-wide ad blocker** designed to run on a single device (typically a Raspberry Pi, though it works on any Linux server). It acts as a "sinkhole" for DNS queries, effectively blocking ads, trackers, and malware domains across your entire home network.

Here is a breakdown of how it works and why it is popular:

### How It Works
Unlike traditional ad blockers that run on individual devices (like browsers or mobile apps), Pi-hole operates at the **DNS level**.
1.  **DNS Interception**: When a device on your network tries to load a website, it first asks the DNS server (usually Pi-hole) for the IP address of that domain.
2.  **Blocking Logic**: Pi-hole checks its list of known ad-serving and tracking domains. If the requested domain is on the blocklist, Pi-hole returns a fake IP address (or an error) instead of the real one.
3.  **Result**: The device cannot connect to the ad server, so the ad never loads. Since the

需要 JSON 的核心场景是：

LLM 的输出不是只给人看，而是还要交给程序继续处理。

常见场景有这些：

1. 调用工具或控制设备

例如智能家居：

{
  "device": "light",
  "action": "turn_on",
  "parameters": {
    "color": "yellow"
  }
}

程序读取后执行开灯。

2. 调用 API

例如让模型生成天气查询参数：

{
  "city": "Tokyo",
  "date": "2026-07-24"
}

后端再拿这些参数请求天气 API。

3. Function Calling / Tool Calling

LLM 判断应该调用哪个函数，并填写参数：

{
  "function": "search_flights",
  "arguments": {
    "from": "Tokyo",
    "to": "Hong Kong"
  }
}
4. 信息抽取

从简历、邮件、合同中抽取固定字段：

{
  "name": "Alex Wang",
  "skills": ["Python", "RAG", "ASR"],
  "experience_years": 5
}

这样数据可以存入数据库。

5. 分类任务

例如判断邮件类型：

{
  "category": "advertisement",
  "confidence": 0.95
}

程序可以根据结果自动删除或分类邮件。

6. RAG 或 Agent 系统

Agent 需要生成下一步行动：

{
  "thought": "需要先搜索资料",
  "action": "retrieve",
  "query": "Whisper fine-tuning Japanese dataset"
}

系统读取 action，再调用检索模块。

7. 前端页面展示

例如生成商品信息：

{
  "title": "MacBook Air",
  "price": 1200,
  "features": ["Lightweight", "Long battery life"]
}

前端可以直接把字段显示到网页或 App 中。

8. 自动化工作流

例如审批系统：

{
  "decision": "approve",
  "reason": "All requirements are satisfied"
}

后面的程序根据 decision 进入下一步。

<a id='3'></a>
## 3 - Guiding the LLM to Output Specific Objects

In this section, you'll explore how to make the LLM generate outputs in specific formats, such as JSON, which can be used by another application. This is a crucial aspect when working with an LLM, as applications often require data in precise formats.

Let's imagine you're automating your home and want to create your personal assistant to control devices like lights and sound systems. The goal is to translate a user's request into a specific format that your home automation server can understand.

In this hypothetical scenario, the format for each action is a JSON structure containing details like this:

```json
{
  "room": "room where the action will occur",
  "object_id": "unique identifier of the targeted object",
  "object_name": "name of the object",
  "action": "action to be performed",
  "parameters": "dictionary containing action-specific parameters"
}
```

For instance, to turn on the office light and set its color to yellow, you should provide the following JSON to the software:

```json
{
  "room": "office",
  "object_id": "152",
  "object_name": "office_light",
  "action": "turn on",
  "parameters": {"color": "yellow"}
}
```


### 3.1 The Old-Fashioned Way  

Let's start with the old-fashioned way by creating a detailed prompt to pass to the LLM.

The following prompt example offers a comprehensive structure. Note how incorporating the JSON format is essential to prevent errors with `f-string` syntax.

**NOTE**: Creating effective prompts often involves a great deal of experimentation. It's a natural part of the creative process to assess the outputs generated by different prompts, identify potential flaws, and make necessary adjustments to refine them.

它反映的是一种比较传统的 Prompt Engineering 方法：

为了让模型输出正确结果，人工把任务规则、设备列表、格式、示例和限制全部写进 Prompt。

这也是图片中所说的：

The Old-Fashioned Way

开发者不使用专门的结构化输出机制，而是依赖一段长 Prompt 告诉模型：

请按照这个 JSON 格式输出，并且不要输出其他内容。

这本质上是一种 基于文本约束模型行为 的方法。

这行代码的作用是：把 Prompt 和模型生成参数整理成一个字典 kwargs，方便后面调用大语言模型。

kwargs = generate_params_dict(
    PROMPT,
    temperature=0.4,
    top_p=0.1
)

可以拆开看。

generate_params_dict(...)

这是一个辅助函数，负责把传入的信息整理成模型 API 所需要的参数格式。

它可能返回类似：

kwargs = {
    "query": PROMPT,
    "temperature": 0.4,
    "top_p": 0.1
}

In [ ]:
def generate_system_call(command):
    PROMPT = f"""
You are an assistant program that converts natural language commands into structured JSON for controlling smart home devices. The JSON should conform to a specific format describing the device, action, and parameters. Here's how you can do it:

**Available Devices and Actions:**

1. **Light**
   - Actions: "turn on", "turn off"
   - Parameters: color, intensity (percentage)

2. **Automatic Lock**
   - Actions: "lock", "unlock"
   - Parameters: None

3. **Sound System (Speaker)**
   - Actions: "play", "pause", "stop", "set volume"
   - Parameters: volume (integer), track (string), playlist_style (string)

4. **TV**
   - Actions: "turn on", "turn off", "change channel", "adjust volume"
   - Parameters: channel (string), volume (integer)

5. **Air Conditioner**
   - Actions: "turn on", "turn off", "set temperature", "adjust fan speed"
   - Parameters: temperature (integer), fan_speed (low/medium/high)

**Rooms and Devices:**
- **Office**
  - Lights: "office_light_1" (ID: 123), "office_light_2" (ID: 321)
  - Automatic Lock: "office_door_lock" (ID: 111)

- **Living Room**
  - Light: "living_room_light" (ID: 222)
  - Speaker: "living_room_speaker" (ID: 223)
  - Air Conditioner: "living_room_airconditioner" (ID: 556)

- **Kitchen**
  - Light: "kitchen_light" (ID: 333)

- **Bedroom**
  - Light: "bedroom_light" (ID: 444)
  - TV: "bedroom_tv" (ID: 445)

- **Bathroom**
  - Light: "bathroom_light" (ID: 555)

**Task:**
Convert the following natural language command into the structured JSON format based on the available devices:

**Input Examples:**

1. "Turn on the office light with ID 123 with blue color and 50% intensity."
   - JSON:
     [
     {{
       "room": "office",
       "object_id": "123",
       "object_name": "office_light_1",
       "action": "turn on",
       "parameters": {{"color": "blue", "intensity": "50%"}}
     }}
     ]

2. "Lock the office door."
   - JSON:
   [
     {{
       "room": "office",
       "object_id": "111",
       "object_name": "office_door_lock",
       "action": "lock",
       "parameters": {{}}
     }}
    ]

2. "Make my living room a cheerful place"
   - JSON:
   [
     {{
       "room": "living_room",
       "object_id": "222",
       "object_name": "living_room_light",
       "action": "turn on",
       "parameters": {{'intensity': '80%', 'color':'yellow'}}
     }},
     {{
       "room": "living_room",
       "object_id": "223",
       "object_name": "living_room_speaker",
       "action": "turn on",
       "parameters": {{'volume': '100', 'playlist_style':'party'}}
     }},

   ]

**Note:**
- Ensure that each JSON object correctly maps the natural command to the appropriate device and action using the listed device ID.
- Use the object ID to differentiate between devices when the room contains multiple similar items.
- You can add more than one parameter in the parameters dictionary.

Using this information, translate the following command into JSON: "{command}". Output a list with all the necessary JSONs.
Always output a list even if there is only one command to be applied, do not output anything else but the desired structure.
"""
    kwargs = generate_params_dict(PROMPT, temperature=0.4, top_p=0.1)
    result = generate_with_single_input(**kwargs)
    return result['content']

In [ ]:
print(generate_system_call("Play a chill playlist very loud"))

[
  {
    "room": "living_room",
    "object_id": "223",
    "object_name": "living_room_speaker",
    "action": "play",
    "parameters": {
      "playlist_style": "chill",
      "volume": "100"
    }
  }
]


In [ ]:
print(generate_system_call("I'm tired today, please make my living room a very cozy ambient, it is really cold today too."))

[
  {
    "room": "living_room",
    "object_id": "222",
    "object_name": "living_room_light",
    "action": "turn on",
    "parameters": {
      "color": "warm white",
      "intensity": "40%"
    }
  },
  {
    "room": "living_room",
    "object_id": "556",
    "object_name": "living_room_airconditioner",
    "action": "set temperature",
    "parameters": {
      "temperature": "24",
      "fan_speed": "low"
    }
  }
]


### 3.2 Using LLM structured output parameter

It is possible to force the LLM to output a JSON using [Pydantic](https://docs.pydantic.dev/latest/) to help it validate the data structure, so you make sure that the output is always a JSON!

Let's see an example below!

2. Pydantic 是什么

Pydantic 是 Python 中用于定义和验证数据结构的工具。

你可以先规定模型输出必须长什么样：

from pydantic import BaseModel

class DeviceAction(BaseModel):
    room: str
    object_id: str
    object_name: str
    action: str
    parameters: dict

In [ ]:
# 从 Pydantic 导入用于定义和验证数据结构的工具
from pydantic import BaseModel, validator, conint, Field

# 从 typing 导入类型标注工具
from typing import Literal, Union, Optional, List

# 导入 Python 自带的 JSON 处理模块
import json


# 定义 LLM 输出的数据结构（schema）
# VoiceNote 继承 BaseModel，因此 Pydantic 会自动验证字段和数据类型
class VoiceNote(BaseModel):

    # 语音笔记的标题
    # 类型必须是字符串 str
    # description 用于告诉 LLM 这个字段应该填写什么内容
    title: str = Field(
        description="A title for the voice note"
    )

    # 语音笔记的一句话摘要
    # 类型必须是字符串 str
    summary: str = Field(
        description="A short one sentence summary of the voice note."
    )

    # 从语音笔记中提取出的待办事项列表
    # list[str] 表示这是一个列表，并且列表中的每个元素都必须是字符串
    actionItems: list[str] = Field(
        description="A list of action items from the voice note"
    )

In [ ]:
transcript = (
        "Good morning! It's 7:00 AM, and I'm just waking up. Today is going to be a busy day, "
        "so let's get started. First, I need to make a quick breakfast. I think I'll have some "
        "scrambled eggs and toast with a cup of coffee. While I'm cooking, I'll also check my "
        "emails to see if there's anything urgent."
    )


messages=[
            {
                "role": "system",
                "content": "The following is a voice message transcript. Only answer in JSON.",
            },
            {
                "role": "user",
                "content": transcript,
            },
        ]

response_format={
            "type": "json_schema",
            "schema": VoiceNote.model_json_schema(),
        }

result = generate_with_multiple_input(messages, response_format = response_format)
result_json = json.loads(result['content'])
print(json.dumps(result_json, indent=2))

{
  "title": "Morning Routine and Breakfast Plan",
  "summary": "User woke up at 7:00 AM, planning a busy day ahead. The morning routine includes preparing scrambled eggs with toast and coffee, while simultaneously checking emails for urgent matters.",
  "actionItems": [
    "Prepare breakfast (scrambled eggs and toast)",
    "Brew coffee",
    "Check emails for urgent items"
  ]
}


Keep it up! You finished the ungraded lab on Prompt Engineering!